In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

DATA_PATH = Path("results/result_sample_shorts_all_for_video_agent_fixed.csv")

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df.shape)

데이터 크기: (200, 75)


In [3]:
numeric_cols = [
    "avg_brightness",
    "avg_blue",
    "avg_green",
    "avg_red",
    "person_ratio",
    "face_ratio",
    "text_ratio",
]

categorical_cols = [
    "production_quality",
    "lighting_style",
    "color_mood",
    "editing_pace",
    "motion_graphic",
    "video_format",
    "first_3sec",
    "background_style",
]

numeric_cols = [col for col in numeric_cols if col in df.columns]
categorical_cols = [col for col in categorical_cols if col in df.columns]

print("수치형 변수:", numeric_cols)
print("범주형 변수:", categorical_cols)

수치형 변수: ['avg_brightness', 'avg_blue', 'avg_green', 'avg_red', 'person_ratio', 'face_ratio', 'text_ratio']
범주형 변수: ['production_quality', 'lighting_style', 'color_mood', 'editing_pace', 'motion_graphic', 'video_format', 'first_3sec', 'background_style']


In [4]:
def cohens_d(x, y):
    """
    두 집단 간 평균 차이의 효과크기 계산
    success - fail 기준으로 계산
    """
    x = x.dropna()
    y = y.dropna()
    
    nx = len(x)
    ny = len(y)
    
    if nx < 2 or ny < 2:
        return np.nan
    
    pooled_std = np.sqrt(
        ((nx - 1) * x.std() ** 2 + (ny - 1) * y.std() ** 2) / (nx + ny - 2)
    )
    
    if pooled_std == 0:
        return np.nan
    
    return (x.mean() - y.mean()) / pooled_std

In [5]:
numeric_test_rows = []

for domain in df["domain"].unique():
    temp = df[df["domain"] == domain].copy()
    
    success_df = temp[temp["success_label"] == "success"]
    fail_df = temp[temp["success_label"] == "fail"]
    
    for col in numeric_cols:
        success_values = success_df[col].dropna()
        fail_values = fail_df[col].dropna()
        
        if len(success_values) < 2 or len(fail_values) < 2:
            continue
        
        # Welch's t-test
        t_stat, t_p = ttest_ind(
            success_values,
            fail_values,
            equal_var=False
        )
        
        # Mann-Whitney U test
        u_stat, u_p = mannwhitneyu(
            success_values,
            fail_values,
            alternative="two-sided"
        )
        
        # 효과크기
        d = cohens_d(success_values, fail_values)
        
        numeric_test_rows.append({
            "domain": domain,
            "feature": col,
            "success_mean": round(success_values.mean(), 3),
            "fail_mean": round(fail_values.mean(), 3),
            "diff_success_minus_fail": round(success_values.mean() - fail_values.mean(), 3),
            "t_test_p": round(t_p, 4),
            "mannwhitney_p": round(u_p, 4),
            "cohens_d": round(d, 3),
            "success_n": len(success_values),
            "fail_n": len(fail_values),
        })

numeric_test_df = pd.DataFrame(numeric_test_rows)

display(
    numeric_test_df.sort_values(
        ["domain", "mannwhitney_p"],
        ascending=[True, True]
    )
)

,domain,feature,success_mean,fail_mean,diff_success_minus_fail,t_test_p,mannwhitney_p,cohens_d,success_n,fail_n
4,FnB,person_ratio,0.786,0.602,0.185,0.0034,0.0008,0.595,48,52
5,FnB,face_ratio,0.649,0.399,0.250,0.0004,0.0009,0.727,48,52
0,FnB,avg_brightness,103.751,124.562,-20.811,0.0202,0.0108,-0.471,48,52
2,FnB,avg_green,99.241,120.713,-21.471,0.0226,0.0150,-0.462,48,52
3,FnB,avg_red,115.789,137.047,-21.258,0.0290,0.0226,-0.441,48,52
1,FnB,avg_blue,93.767,111.943,-18.176,0.0551,0.0473,-0.388,48,52
6,FnB,text_ratio,0.712,0.652,0.060,0.2040,0.1618,0.256,48,52
10,IT,avg_red,112.947,97.780,15.167,0.1087,0.0871,0.325,42,58
8,IT,avg_blue,119.381,103.477,15.904,0.1213,0.1253,0.311,42,58
7,IT,avg_brightness,112.247,97.767,14.480,0.1220,0.1614,0.312,42,58


In [6]:
key_numeric_features = [
    "person_ratio",
    "face_ratio",
    "text_ratio",
]

display(
    numeric_test_df[
        numeric_test_df["feature"].isin(key_numeric_features)
    ].sort_values(["domain", "feature"])
)

,domain,feature,success_mean,fail_mean,diff_success_minus_fail,t_test_p,mannwhitney_p,cohens_d,success_n,fail_n
5,FnB,face_ratio,0.649,0.399,0.250,0.0004,0.0009,0.727,48,52
4,FnB,person_ratio,0.786,0.602,0.185,0.0034,0.0008,0.595,48,52
6,FnB,text_ratio,0.712,0.652,0.060,0.2040,0.1618,0.256,48,52
12,IT,face_ratio,0.524,0.641,-0.118,0.0914,0.1727,-0.355,42,58
11,IT,person_ratio,0.621,0.734,-0.113,0.1012,0.2531,-0.350,42,58
13,IT,text_ratio,0.807,0.769,0.037,0.2322,0.4118,0.233,42,58


In [7]:
def cramers_v(confusion_matrix):
    """
    카이제곱 검정의 효과크기 계산
    범주형 변수 간 관계 강도 측정
    """
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    
    if n == 0:
        return np.nan
    
    return np.sqrt((chi2 / n) / min(k - 1, r - 1))

In [8]:
category_test_rows = []

for domain in df["domain"].unique():
    temp = df[df["domain"] == domain].copy()
    
    for col in categorical_cols:
        table = pd.crosstab(temp["success_label"], temp[col])
        
        if table.shape[0] < 2 or table.shape[1] < 2:
            continue
        
        chi2, p, dof, expected = chi2_contingency(table)
        cv = cramers_v(table)
        
        category_test_rows.append({
            "domain": domain,
            "feature": col,
            "chi2_p": round(p, 4),
            "cramers_v": round(cv, 3),
            "n_categories": table.shape[1],
            "success_n": table.loc["success"].sum() if "success" in table.index else 0,
            "fail_n": table.loc["fail"].sum() if "fail" in table.index else 0,
        })

category_test_df = pd.DataFrame(category_test_rows)

display(
    category_test_df.sort_values(
        ["domain", "chi2_p"],
        ascending=[True, True]
    )
)

,domain,feature,chi2_p,cramers_v,n_categories,success_n,fail_n
4,FnB,motion_graphic,0.0022,0.350,3,48,52
6,FnB,first_3sec,0.0197,0.366,6,48,52
5,FnB,video_format,0.1155,0.409,12,48,52
2,FnB,color_mood,0.2152,0.211,4,48,52
3,FnB,editing_pace,0.4353,0.165,4,48,52
0,FnB,production_quality,0.4391,0.128,3,48,52
7,FnB,background_style,0.6003,0.137,4,48,52
1,FnB,lighting_style,0.6161,0.098,3,48,52
12,IT,motion_graphic,0.0110,0.254,2,42,58
14,IT,first_3sec,0.0787,0.261,4,42,58


In [9]:
key_categorical_features = [
    "first_3sec",
    "motion_graphic",
    "editing_pace",
    "video_format",
]

display(
    category_test_df[
        category_test_df["feature"].isin(key_categorical_features)
    ].sort_values(["domain", "chi2_p"])
)

,domain,feature,chi2_p,cramers_v,n_categories,success_n,fail_n
4,FnB,motion_graphic,0.0022,0.350,3,48,52
6,FnB,first_3sec,0.0197,0.366,6,48,52
5,FnB,video_format,0.1155,0.409,12,48,52
3,FnB,editing_pace,0.4353,0.165,4,48,52
12,IT,motion_graphic,0.0110,0.254,2,42,58
14,IT,first_3sec,0.0787,0.261,4,42,58
13,IT,video_format,0.1017,0.430,13,42,58
11,IT,editing_pace,0.2295,0.237,5,42,58


In [10]:
def make_category_ratio_table(df, category_col, group_cols=["domain", "success_label"]):
    count_df = (
        df.groupby(group_cols + [category_col])
          .size()
          .reset_index(name="count")
    )
    
    total_df = (
        df.groupby(group_cols)
          .size()
          .reset_index(name="total")
    )
    
    result = count_df.merge(total_df, on=group_cols, how="left")
    result["ratio"] = (result["count"] / result["total"]).round(3)
    
    return result

In [11]:
def make_success_fail_category_diff(df, category_col):
    ratio_df = make_category_ratio_table(df, category_col)
    
    pivot = ratio_df.pivot_table(
        index=["domain", category_col],
        columns="success_label",
        values="ratio",
        fill_value=0
    ).reset_index()
    
    if "success" not in pivot.columns:
        pivot["success"] = 0
    if "fail" not in pivot.columns:
        pivot["fail"] = 0
    
    pivot["diff_success_minus_fail"] = (pivot["success"] - pivot["fail"]).round(3)
    
    return pivot.sort_values(
        ["domain", "diff_success_minus_fail"],
        ascending=[True, False]
    )

In [12]:
category_diff_dict = {}

for col in key_categorical_features:
    if col not in df.columns:
        continue
    
    diff_table = make_success_fail_category_diff(df, col)
    category_diff_dict[col] = diff_table
    
    print("=" * 80)
    print(col)
    display(diff_table)

first_3sec


success_label,domain,first_3sec,fail,success,diff_success_minus_fail
2,FnB,인물,0.212,0.521,0.309
3,FnB,장면,0.019,0.042,0.023
4,FnB,제품,0.173,0.167,-0.006
1,FnB,음식,0.038,0.021,-0.017
0,FnB,기업 로고,0.019,0.000,-0.019
5,FnB,텍스트,0.538,0.250,-0.288
9,IT,텍스트,0.621,0.833,0.212
8,IT,제품,0.034,0.000,-0.034
7,IT,장면,0.052,0.000,-0.052
6,IT,인물,0.293,0.167,-0.126


motion_graphic


success_label,domain,motion_graphic,fail,success,diff_success_minus_fail
0,FnB,보조적,0.538,0.854,0.316
1,FnB,없음,0.019,0.021,0.002
2,FnB,핵심요소,0.442,0.125,-0.317
4,IT,핵심요소,0.276,0.548,0.272
3,IT,보조적,0.724,0.452,-0.272


editing_pace


success_label,domain,editing_pace,fail,success,diff_success_minus_fail
3,FnB,빠름,0.462,0.625,0.163
0,FnB,매우 느림,0.038,0.021,-0.017
2,FnB,보통,0.115,0.083,-0.032
1,FnB,매우 빠름,0.385,0.271,-0.114
6,IT,매우 빠름,0.172,0.262,0.090
8,IT,빠름,0.466,0.548,0.082
4,IT,느림,0.017,0.000,-0.017
5,IT,매우 느림,0.069,0.000,-0.069
7,IT,보통,0.276,0.190,-0.086


video_format


success_label,domain,video_format,fail,success,diff_success_minus_fail
7,FnB,웹예능,0.019,0.167,0.148
10,FnB,제품리뷰,0.019,0.083,0.064
6,FnB,웹드라마,0.000,0.062,0.062
8,FnB,이벤트/행사,0.077,0.104,0.027
2,FnB,시설소개,0.019,0.021,0.002
4,FnB,에피소드소개,0.019,0.021,0.002
3,FnB,애니메이션,0.038,0.021,-0.017
9,FnB,인터뷰,0.038,0.021,-0.017
1,FnB,브이로그,0.019,0.000,-0.019
5,FnB,요리/레시피,0.019,0.000,-0.019


In [13]:
# FnB 수치형 핵심 변수
display(
    numeric_test_df[
        (numeric_test_df["domain"] == "FnB") &
        (numeric_test_df["feature"].isin(["person_ratio", "face_ratio"]))
    ]
)

# FnB first_3sec 검정 결과
display(
    category_test_df[
        (category_test_df["domain"] == "FnB") &
        (category_test_df["feature"] == "first_3sec")
    ]
)

# FnB first_3sec 범주별 차이
display(
    category_diff_dict["first_3sec"][
        category_diff_dict["first_3sec"]["domain"] == "FnB"
    ]
)

,domain,feature,success_mean,fail_mean,diff_success_minus_fail,t_test_p,mannwhitney_p,cohens_d,success_n,fail_n
4,FnB,person_ratio,0.786,0.602,0.185,0.0034,0.0008,0.595,48,52
5,FnB,face_ratio,0.649,0.399,0.250,0.0004,0.0009,0.727,48,52


,domain,feature,chi2_p,cramers_v,n_categories,success_n,fail_n
6,FnB,first_3sec,0.0197,0.366,6,48,52


success_label,domain,first_3sec,fail,success,diff_success_minus_fail
2,FnB,인물,0.212,0.521,0.309
3,FnB,장면,0.019,0.042,0.023
4,FnB,제품,0.173,0.167,-0.006
1,FnB,음식,0.038,0.021,-0.017
0,FnB,기업 로고,0.019,0.000,-0.019
5,FnB,텍스트,0.538,0.250,-0.288


In [14]:
# IT text_ratio 검정 결과
display(
    numeric_test_df[
        (numeric_test_df["domain"] == "IT") &
        (numeric_test_df["feature"] == "text_ratio")
    ]
)

# IT first_3sec, motion_graphic 검정 결과
display(
    category_test_df[
        (category_test_df["domain"] == "IT") &
        (category_test_df["feature"].isin(["first_3sec", "motion_graphic"]))
    ]
)

# IT first_3sec 범주별 차이
display(
    category_diff_dict["first_3sec"][
        category_diff_dict["first_3sec"]["domain"] == "IT"
    ]
)

# IT motion_graphic 범주별 차이
display(
    category_diff_dict["motion_graphic"][
        category_diff_dict["motion_graphic"]["domain"] == "IT"
    ]
)

,domain,feature,success_mean,fail_mean,diff_success_minus_fail,t_test_p,mannwhitney_p,cohens_d,success_n,fail_n
13,IT,text_ratio,0.807,0.769,0.037,0.2322,0.4118,0.233,42,58


,domain,feature,chi2_p,cramers_v,n_categories,success_n,fail_n
12,IT,motion_graphic,0.0110,0.254,2,42,58
14,IT,first_3sec,0.0787,0.261,4,42,58


success_label,domain,first_3sec,fail,success,diff_success_minus_fail
9,IT,텍스트,0.621,0.833,0.212
8,IT,제품,0.034,0.000,-0.034
7,IT,장면,0.052,0.000,-0.052
6,IT,인물,0.293,0.167,-0.126


success_label,domain,motion_graphic,fail,success,diff_success_minus_fail
4,IT,핵심요소,0.276,0.548,0.272
3,IT,보조적,0.724,0.452,-0.272


In [15]:
# editing_pace 검정 결과
display(
    category_test_df[
        category_test_df["feature"] == "editing_pace"
    ]
)

# editing_pace 범주별 차이
display(category_diff_dict["editing_pace"])

,domain,feature,chi2_p,cramers_v,n_categories,success_n,fail_n
3,FnB,editing_pace,0.4353,0.165,4,48,52
11,IT,editing_pace,0.2295,0.237,5,42,58


success_label,domain,editing_pace,fail,success,diff_success_minus_fail
3,FnB,빠름,0.462,0.625,0.163
0,FnB,매우 느림,0.038,0.021,-0.017
2,FnB,보통,0.115,0.083,-0.032
1,FnB,매우 빠름,0.385,0.271,-0.114
6,IT,매우 빠름,0.172,0.262,0.090
8,IT,빠름,0.466,0.548,0.082
4,IT,느림,0.017,0.000,-0.017
5,IT,매우 느림,0.069,0.000,-0.069
7,IT,보통,0.276,0.190,-0.086


# 정규성 검정


In [16]:
from scipy.stats import shapiro
import pandas as pd
import numpy as np

normality_rows = []

for domain in df["domain"].unique():
    for label in ["success", "fail"]:
        temp = df[
            (df["domain"] == domain) &
            (df["success_label"] == label)
        ].copy()
        
        for col in numeric_cols:
            values = temp[col].dropna()
            
            # Shapiro-Wilk는 최소 3개 이상 필요
            if len(values) < 3:
                normality_rows.append({
                    "domain": domain,
                    "success_label": label,
                    "feature": col,
                    "n": len(values),
                    "shapiro_stat": np.nan,
                    "shapiro_p": np.nan,
                    "normality_result": "표본 부족"
                })
                continue
            
            stat, p = shapiro(values)
            
            normality_rows.append({
                "domain": domain,
                "success_label": label,
                "feature": col,
                "n": len(values),
                "shapiro_stat": round(stat, 4),
                "shapiro_p": round(p, 4),
                "normality_result": "정규성 만족 가능" if p >= 0.05 else "정규성 불만족 가능"
            })

normality_df = pd.DataFrame(normality_rows)

display(
    normality_df.sort_values(
        ["domain", "feature", "success_label"]
    )
)

,domain,success_label,feature,n,shapiro_stat,shapiro_p,normality_result
8,FnB,fail,avg_blue,52,0.9779,0.4423,정규성 만족 가능
1,FnB,success,avg_blue,48,0.9611,0.1123,정규성 만족 가능
7,FnB,fail,avg_brightness,52,0.9645,0.1221,정규성 만족 가능
0,FnB,success,avg_brightness,48,0.9677,0.2052,정규성 만족 가능
9,FnB,fail,avg_green,52,0.9713,0.2405,정규성 만족 가능
2,FnB,success,avg_green,48,0.9721,0.3041,정규성 만족 가능
10,FnB,fail,avg_red,52,0.9513,0.0332,정규성 불만족 가능
3,FnB,success,avg_red,48,0.9628,0.1313,정규성 만족 가능
12,FnB,fail,face_ratio,52,0.8172,0.0000,정규성 불만족 가능
5,FnB,success,face_ratio,48,0.8370,0.0000,정규성 불만족 가능


In [17]:
normality_summary = (
    normality_df
    .groupby(["domain", "feature", "normality_result"])
    .size()
    .reset_index(name="count")
)

display(normality_summary)

,domain,feature,normality_result,count
0,FnB,avg_blue,정규성 만족 가능,2
1,FnB,avg_brightness,정규성 만족 가능,2
2,FnB,avg_green,정규성 만족 가능,2
3,FnB,avg_red,정규성 만족 가능,1
4,FnB,avg_red,정규성 불만족 가능,1
5,FnB,face_ratio,정규성 불만족 가능,2
6,FnB,person_ratio,정규성 불만족 가능,2
7,FnB,text_ratio,정규성 불만족 가능,2
8,IT,avg_blue,정규성 만족 가능,1
9,IT,avg_blue,정규성 불만족 가능,1


In [18]:
from scipy.stats import levene

variance_rows = []

for domain in df["domain"].unique():
    temp = df[df["domain"] == domain].copy()
    
    success_df = temp[temp["success_label"] == "success"]
    fail_df = temp[temp["success_label"] == "fail"]
    
    for col in numeric_cols:
        success_values = success_df[col].dropna()
        fail_values = fail_df[col].dropna()
        
        # Levene test는 각 그룹 최소 2개 이상 필요
        if len(success_values) < 2 or len(fail_values) < 2:
            variance_rows.append({
                "domain": domain,
                "feature": col,
                "success_n": len(success_values),
                "fail_n": len(fail_values),
                "levene_stat": np.nan,
                "levene_p": np.nan,
                "variance_result": "표본 부족"
            })
            continue
        
        stat, p = levene(success_values, fail_values)
        
        variance_rows.append({
            "domain": domain,
            "feature": col,
            "success_n": len(success_values),
            "fail_n": len(fail_values),
            "levene_stat": round(stat, 4),
            "levene_p": round(p, 4),
            "variance_result": "등분산 가정 가능" if p >= 0.05 else "등분산 가정 어려움"
        })

variance_df = pd.DataFrame(variance_rows)

display(
    variance_df.sort_values(["domain", "feature"])
)

,domain,feature,success_n,fail_n,levene_stat,levene_p,variance_result
1,FnB,avg_blue,48,52,0.0096,0.9222,등분산 가정 가능
0,FnB,avg_brightness,48,52,0.2169,0.6425,등분산 가정 가능
2,FnB,avg_green,48,52,0.5227,0.4714,등분산 가정 가능
3,FnB,avg_red,48,52,0.8203,0.3673,등분산 가정 가능
5,FnB,face_ratio,48,52,11.6313,0.0009,등분산 가정 어려움
4,FnB,person_ratio,48,52,4.4839,0.0367,등분산 가정 어려움
6,FnB,text_ratio,48,52,0.0222,0.8820,등분산 가정 가능
8,IT,avg_blue,42,58,2.3921,0.1252,등분산 가정 가능
7,IT,avg_brightness,42,58,2.1094,0.1496,등분산 가정 가능
9,IT,avg_green,42,58,1.7469,0.1893,등분산 가정 가능


In [19]:
display(
    variance_df[
        variance_df["variance_result"] == "등분산 가정 어려움"
    ].sort_values(["domain", "feature"])
)

,domain,feature,success_n,fail_n,levene_stat,levene_p,variance_result
5,FnB,face_ratio,48,52,11.6313,0.0009,등분산 가정 어려움
4,FnB,person_ratio,48,52,4.4839,0.0367,등분산 가정 어려움


In [20]:
assumption_summary = variance_df.merge(
    normality_df.groupby(["domain", "feature"])["normality_result"]
    .apply(lambda x: ", ".join(sorted(set(x))))
    .reset_index(),
    on=["domain", "feature"],
    how="left"
)

assumption_summary = assumption_summary[
    [
        "domain",
        "feature",
        "success_n",
        "fail_n",
        "normality_result",
        "levene_p",
        "variance_result"
    ]
]

display(assumption_summary)

,domain,feature,success_n,fail_n,normality_result,levene_p,variance_result
0,FnB,avg_brightness,48,52,정규성 만족 가능,0.6425,등분산 가정 가능
1,FnB,avg_blue,48,52,정규성 만족 가능,0.9222,등분산 가정 가능
2,FnB,avg_green,48,52,정규성 만족 가능,0.4714,등분산 가정 가능
3,FnB,avg_red,48,52,"정규성 만족 가능, 정규성 불만족 가능",0.3673,등분산 가정 가능
4,FnB,person_ratio,48,52,정규성 불만족 가능,0.0367,등분산 가정 어려움
5,FnB,face_ratio,48,52,정규성 불만족 가능,0.0009,등분산 가정 어려움
6,FnB,text_ratio,48,52,정규성 불만족 가능,0.8820,등분산 가정 가능
7,IT,avg_brightness,42,58,"정규성 만족 가능, 정규성 불만족 가능",0.1496,등분산 가정 가능
8,IT,avg_blue,42,58,"정규성 만족 가능, 정규성 불만족 가능",0.1252,등분산 가정 가능
9,IT,avg_green,42,58,정규성 불만족 가능,0.1893,등분산 가정 가능
